# Toy Policy Gate

The policy gate is intentionally simple. It should be boring, explicit, and hard to bypass.

In [3]:
POLICY = {
    "paper_only": True,
    "symbol_allowlist": {"SPY", "QQQ", "AAPL", "MSFT", "NVDA"},
    "max_notional_per_trade": 500.0,
    "min_confidence_to_trade": 0.58,
}

def evaluate_policy(symbol, action, qty, latest_price, confidence, paper_mode=True):
    reasons = []
    notional = qty * latest_price

    if POLICY["paper_only"] and not paper_mode:
        reasons.append("paper trading is required")
    if symbol not in POLICY["symbol_allowlist"]:
        reasons.append("symbol is not allowlisted")
    if action == "hold":
        reasons.append("model selected hold")
    if notional > POLICY["max_notional_per_trade"]:
        reasons.append("order is too large")
    if confidence < POLICY["min_confidence_to_trade"]:
        reasons.append("confidence is too low")

    if reasons:
        return {"status": "rejected", "final_action": "hold", "reasons": reasons}
    return {"status": "approved", "final_action": action, "reasons": []}

examples = [
    ("SPY", "buy", 1, 100.0, 0.61, True),
    ("SPY", "buy", 10, 100.0, 0.61, True),
    ("TSLA", "buy", 1, 100.0, 0.61, True),
    ("SPY", "hold", 1, 100.0, 0.70, True),
    ("SPY", "buy", 1, 100.0, 0.51, True),
]

[evaluate_policy(*example) for example in examples]

[{'status': 'approved', 'final_action': 'buy', 'reasons': []},
 {'status': 'rejected',
  'final_action': 'hold',
  'reasons': ['order is too large']},
 {'status': 'rejected',
  'final_action': 'hold',
  'reasons': ['symbol is not allowlisted']},
 {'status': 'rejected',
  'final_action': 'hold',
  'reasons': ['model selected hold']},
 {'status': 'rejected',
  'final_action': 'hold',
  'reasons': ['confidence is too low']}]

The real orchestrator adds database checks too: daily trade count and per-symbol cooldown.